In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import requests
from time import sleep
import os
import re

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'CD BCCO'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running CD BCCO Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
# BCC pages are static; loaded with requests.get(), no Chrome driver needed.
driver = None

In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
           'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
           'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
           'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
           'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
           'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

regdict = {
    regulatorName + ' 1': 'https://www.bcc.cd/surveillance-des-intermediaires-financiers/intermediaires-financiers-assujettis/etablissements-de-credit/banques-agreees',
    regulatorName + ' 2': 'https://www.bcc.cd/surveillance-des-intermediaires-financiers/intermediaires-financiers-assujettis/etablissements-de-credit/imf',
    regulatorName + ' 3': 'https://www.bcc.cd/surveillance-des-intermediaires-financiers/intermediaires-financiers-assujettis/etablissements-de-credit/coopec',
    regulatorName + ' 4': 'https://www.bcc.cd/surveillance-des-intermediaires-financiers/intermediaires-financiers-assujettis/etablissements-de-credit/ifs',
    regulatorName + ' 5': 'https://www.bcc.cd/surveillance-des-intermediaires-financiers/intermediaires-financiers-assujettis/etablissements-de-credit/cec',
    regulatorName + ' 6': 'https://www.bcc.cd/surveillance-des-intermediaires-financiers/intermediaires-financiers-assujettis/etablissements-de-credit/societes-financieres/ee',
    regulatorName + ' 7': 'https://www.bcc.cd/surveillance-des-intermediaires-financiers/intermediaires-financiers-assujettis/etablissements-de-credit/societes-financieres/au',
}

Typology = {
    regulatorName + ' 1': 'List of Banks',
    regulatorName + ' 2': 'List of Microfinance Institutions',
    regulatorName + ' 3': 'List of Savings and Credit Cooperatives',
    regulatorName + ' 4': 'List of Specialized Financial Institutions',
    regulatorName + ' 5': 'List of Savings and Credit Banks',
    regulatorName + ' 6': 'List of Electronic Money Issuing Institutions',
    regulatorName + ' 7': 'List of Other Financial Companies',
}

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key in sqldict:
        if len(sqldict[key]) != maxlen:
            sqldict[key] = sqldict[key] + [''] * (maxlen - len(sqldict[key]))
    return sqldict


def clean_text(text):
    if text is None:
        return ''
    text = text.replace('\xa0', ' ').replace('\u200b', ' ')
    return re.sub(r'\s+', ' ', text).strip(' ;,.-')


def fetch_html(url, timeout=120):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'fr,en-US;q=0.9,en;q=0.8'
    }
    try:
        requests.packages.urllib3.disable_warnings()
    except Exception:
        pass
    response = requests.get(url, headers=headers, timeout=timeout, verify=False)
    response.raise_for_status()
    response.encoding = response.encoding or response.apparent_encoding or 'utf-8'
    return response.text


def append_entity(reg, list_name, name_val, address_val, city_val='', phone_val='', email_val=''):
    sqldict['Name'].append(name_val)
    sqldict['Address_1'].append(address_val)
    sqldict['City'].append(city_val)
    sqldict['Cntry'].append('CD')
    sqldict['Phone'].append(phone_val)
    sqldict['Email'].append(email_val)
    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegCtry'].append('CD')
    sqldict['RegCode'].append('BCCO')
    sqldict['ListCode'].append(reg.split(' ')[-1])
    sqldict['ListLanguage'].append('French')
    sqldict['RegulationType'].append('Regulated')
    sqldict['ListName'].append(list_name)
    bourange_same_length_array(sqldict)


def header_map(table):
    """Return {normalized_header: column_index} from the first row that contains <th>."""
    header_row = None
    for tr in table.find_all('tr'):
        if tr.find('th'):
            header_row = tr
            break
    if header_row is None:
        return {}
    mapping = {}
    for i, th in enumerate(header_row.find_all(['th', 'td'])):
        key = clean_text(th.get_text(' ', strip=True)).lower()
        mapping[key] = i
    return mapping


def find_col(hmap, *needles):
    """Return the first column index whose header contains any of the needles."""
    for key, idx in hmap.items():
        for n in needles:
            if n.lower() in key:
                return idx
    return None

In [6]:
#------------------------------------------------ List 1 : Banques Agreees ----------------------------------------
# Table has 5 columns: Denomination | Autorisation | Siege Social | Provinces | Points d'Exploitation
# (rowspan on the first 3). One row per entity is the first physical <tr> (the one carrying all 5 <td>).
# City = Provinces cell. Address_1 = Points cell with leading "Siege Social/..." segment stripped.
reg = regulatorName + ' 1'
url = regdict[reg]
list_name = Typology[reg]
print(f'Working with {reg} - {list_name}')

soup = BeautifulSoup(fetch_html(url), 'html.parser')
table = soup.find('table')
count = 0
for tr in table.find_all('tr'):
    if tr.find('th'):
        continue
    cells = tr.find_all('td')
    if len(cells) < 5:
        continue
    name_val = clean_text(cells[0].get_text(' ', strip=True))
    if not name_val:
        continue
    city_val = clean_text(cells[3].get_text(' ', strip=True))
    points_text = cells[4].get_text(' ', strip=True)
    parts = [p.strip() for p in points_text.split(',') if p.strip()]
    parts = [p for p in parts if not re.match(r'(?i)si[eè]ge\s*social', p)]
    address_val = clean_text(', '.join(parts))
    append_entity(reg, list_name, name_val, address_val, city_val)
    count += 1
print(f'[INFO] : {count} entities on {reg}')

Working with CD BCCO 1 - List of Banks
[INFO] : 15 entities on CD BCCO 1


In [7]:
#------------------------------------------------ List 2 : Microfinance Institutions ----------------------------------------
# Columns mapped by header text:
#   Denomination -> Name | Rayon d'action -> City | Adresse Physique -> Address_1
#   Numero de Tel. -> Phone | E-MAIL -> Email
reg = regulatorName + ' 2'
url = regdict[reg]
list_name = Typology[reg]
print(f'Working with {reg} - {list_name}')

soup = BeautifulSoup(fetch_html(url), 'html.parser')
table = soup.find('table')
hmap = header_map(table)
i_name = find_col(hmap, 'dénomination', 'denomination')
i_city = find_col(hmap, "rayon d'action", 'rayon')
i_addr = find_col(hmap, 'adresse physique', 'adresse')
i_phone = find_col(hmap, 'numéro de tél', 'tél', 'tel')
i_email = find_col(hmap, 'e-mail', 'email', 'mail')

seen = set()
for tr in table.find_all('tr'):
    if tr.find('th'):
        continue
    cells = tr.find_all('td')
    if i_name is None or len(cells) <= i_name:
        continue
    name_val = clean_text(cells[i_name].get_text(' ', strip=True))
    if not name_val or name_val in seen:
        continue
    def _cell(i):
        return clean_text(cells[i].get_text(' ', strip=True)) if (i is not None and i < len(cells)) else ''
    city_val = _cell(i_city)
    address_val = _cell(i_addr)
    phone_val = _cell(i_phone)
    email_val = _cell(i_email)
    seen.add(name_val)
    append_entity(reg, list_name, name_val, address_val, city_val, phone_val, email_val)
print(f'[INFO] : {len(seen)} entities on {reg}')

Working with CD BCCO 2 - List of Microfinance Institutions
[INFO] : 23 entities on CD BCCO 2


In [8]:
#------------------------------------------------ List 3 : Savings and Credit Cooperatives ----------------------------------------
# Same header-driven mapping as List 2:
#   Denomination -> Name | Rayon d'action -> City | Adresse Physique -> Address_1
#   Numero de Tel. -> Phone | E-MAIL -> Email
reg = regulatorName + ' 3'
url = regdict[reg]
list_name = Typology[reg]
print(f'Working with {reg} - {list_name}')

soup = BeautifulSoup(fetch_html(url), 'html.parser')
table = soup.find('table')
hmap = header_map(table)
i_name = find_col(hmap, 'dénomination', 'denomination')
i_city = find_col(hmap, "rayon d'action", 'rayon')
i_addr = find_col(hmap, 'adresse physique', 'adresse')
i_phone = find_col(hmap, 'numéro de tél', 'tél', 'tel')
i_email = find_col(hmap, 'e-mail', 'email', 'mail')

seen = set()
for tr in table.find_all('tr'):
    if tr.find('th'):
        continue
    cells = tr.find_all('td')
    if i_name is None or len(cells) <= i_name:
        continue
    name_val = clean_text(cells[i_name].get_text(' ', strip=True))
    if not name_val or name_val in seen:
        continue
    def _cell(i):
        return clean_text(cells[i].get_text(' ', strip=True)) if (i is not None and i < len(cells)) else ''
    city_val = _cell(i_city)
    address_val = _cell(i_addr)
    phone_val = _cell(i_phone)
    email_val = _cell(i_email)
    seen.add(name_val)
    append_entity(reg, list_name, name_val, address_val, city_val, phone_val, email_val)
print(f'[INFO] : {len(seen)} entities on {reg}')

Working with CD BCCO 3 - List of Savings and Credit Cooperatives
[INFO] : 78 entities on CD BCCO 3


In [9]:
#------------------------------------------------ List 4 : Specialized Financial Institutions ----------------------------------------
reg = regulatorName + ' 4'
url = regdict[reg]
list_name = Typology[reg]
print(f'Working with {reg} - {list_name}')

soup = BeautifulSoup(fetch_html(url), 'html.parser')
table = soup.find('table')
seen = set()
for tr in table.find_all('tr'):
    if tr.find('th'):
        continue
    cells = tr.find_all('td')
    if len(cells) < 3:
        continue
    name_val = clean_text(cells[0].get_text(' ', strip=True))
    siege_val = clean_text(cells[2].get_text(' ', strip=True))
    if not name_val or name_val in seen:
        continue
    seen.add(name_val)
    append_entity(reg, list_name, name_val, siege_val)
print(f'[INFO] : {len(seen)} entities on {reg}')

Working with CD BCCO 4 - List of Specialized Financial Institutions
[INFO] : 3 entities on CD BCCO 4


In [10]:
#------------------------------------------------ List 5 : Savings and Credit Banks ----------------------------------------
reg = regulatorName + ' 5'
url = regdict[reg]
list_name = Typology[reg]
print(f'Working with {reg} - {list_name}')

soup = BeautifulSoup(fetch_html(url), 'html.parser')
table = soup.find('table')
seen = set()
for tr in table.find_all('tr'):
    if tr.find('th'):
        continue
    cells = tr.find_all('td')
    if len(cells) < 3:
        continue
    name_val = clean_text(cells[0].get_text(' ', strip=True))
    siege_val = clean_text(cells[2].get_text(' ', strip=True))
    if not name_val or name_val in seen:
        continue
    seen.add(name_val)
    append_entity(reg, list_name, name_val, siege_val)
print(f'[INFO] : {len(seen)} entities on {reg}')

Working with CD BCCO 5 - List of Savings and Credit Banks
[INFO] : 1 entities on CD BCCO 5


In [11]:
#------------------------------------------------ List 6 : Electronic Money Issuing Institutions ----------------------------------------
reg = regulatorName + ' 6'
url = regdict[reg]
list_name = Typology[reg]
print(f'Working with {reg} - {list_name}')

soup = BeautifulSoup(fetch_html(url), 'html.parser')
table = soup.find('table')
seen = set()
for tr in table.find_all('tr'):
    if tr.find('th'):
        continue
    cells = tr.find_all('td')
    if len(cells) < 3:
        continue
    name_val = clean_text(cells[0].get_text(' ', strip=True))
    siege_val = clean_text(cells[2].get_text(' ', strip=True))
    if not name_val or name_val in seen:
        continue
    seen.add(name_val)
    append_entity(reg, list_name, name_val, siege_val)
print(f'[INFO] : {len(seen)} entities on {reg}')

Working with CD BCCO 6 - List of Electronic Money Issuing Institutions
[INFO] : 4 entities on CD BCCO 6


In [12]:
#------------------------------------------------ List 7 : Other Financial Companies ----------------------------------------
reg = regulatorName + ' 7'
url = regdict[reg]
list_name = Typology[reg]
print(f'Working with {reg} - {list_name}')

soup = BeautifulSoup(fetch_html(url), 'html.parser')
table = soup.find('table')
seen = set()
for tr in table.find_all('tr'):
    if tr.find('th'):
        continue
    cells = tr.find_all('td')
    if len(cells) < 3:
        continue
    name_val = clean_text(cells[0].get_text(' ', strip=True))
    siege_val = clean_text(cells[2].get_text(' ', strip=True))
    if not name_val or name_val in seen:
        continue
    seen.add(name_val)
    append_entity(reg, list_name, name_val, siege_val)
print(f'[INFO] : {len(seen)} entities on {reg}')

Working with CD BCCO 7 - List of Other Financial Companies
[INFO] : 1 entities on CD BCCO 7


In [13]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, 'SQL Ready', index=False)
if driver is not None:
    driver.quit()
sleep(3)
print(f"[INFO] : Excel file '{filename}' saved successfully")

C:\Users\wuj1\AppData\Local\Temp\8\ipykernel_19904\1214484887.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


[INFO] : Excel file 'CD BCCO SQL Ready 2026-05-18 18.47.22.xlsx' saved successfully


In [14]:
#------------------------------------------------ Data Integrity & Consistency Check ----------------------------------------
print("=" * 80)
print("DATA INTEGRITY & CONSISTENCY VERIFICATION")
print("=" * 80)

print("\n1. DATAFRAME SHAPE:")
print(f"   Total rows collected: {len(df)}")
print(f"   Total columns: {len(df.columns)}")

print("\n2. DATA DISTRIBUTION BY LIST:")
if len(df) > 0:
    list_summary = df.groupby('ListCode').agg({
        'Name': 'count',
        'ListName': 'first',
        'RegCtry': 'first',
        'RegCode': 'first'
    }).rename(columns={'Name': 'Count'})
    print(list_summary)
else:
    print('   NO DATA COLLECTED')

print("\n3. REGCTRY & REGCODE VALIDATION:")
if len(df) > 0:
    regctry_values = df['RegCtry'].unique()
    regcode_values = df['RegCode'].unique()
    language_values = df['ListLanguage'].unique()
    regctry_status = "PASS" if all(v == 'CD' for v in regctry_values) else "FAIL"
    regcode_status = "PASS" if all(v == 'BCCO' for v in regcode_values) else "FAIL"
    language_status = "PASS" if all(v == 'French' for v in language_values) else "FAIL"
    print(f"   RegCtry values: {regctry_values} {regctry_status}")
    print(f"   RegCode values: {regcode_values} {regcode_status}")
    print(f"   ListLanguage values: {language_values} {language_status}")

print("\n4. KEY FIELDS VALIDATION:")
for col in ['Name', 'Address_1']:
    filled = (df[col].astype(str).str.strip() != '').sum() if len(df) > 0 else 0
    print(f"   {col} field filled: {filled}/{len(df)}")

print("\n5. SAMPLE DATA (first 5 rows):")
if len(df) > 0:
    print(df[['Name', 'Address_1', 'ListCode', 'ListName']].head(5).to_string())
else:
    print('   NO DATA COLLECTED')

print("\n" + "=" * 80)
print("SUMMARY:")
print("=" * 80)
print(f"Total rows in DataFrame: {len(df)}")
print(f"Distinct ListCodes: {df['ListCode'].nunique() if len(df) > 0 else 0}/7")
print("=" * 80)

DATA INTEGRITY & CONSISTENCY VERIFICATION

1. DATAFRAME SHAPE:
   Total rows collected: 125
   Total columns: 44

2. DATA DISTRIBUTION BY LIST:
          Count                                       ListName RegCtry RegCode
ListCode                                                                      
1            15                                  List of Banks      CD    BCCO
2            23              List of Microfinance Institutions      CD    BCCO
3            78        List of Savings and Credit Cooperatives      CD    BCCO
4             3     List of Specialized Financial Institutions      CD    BCCO
5             1               List of Savings and Credit Banks      CD    BCCO
6             4  List of Electronic Money Issuing Institutions      CD    BCCO
7             1              List of Other Financial Companies      CD    BCCO

3. REGCTRY & REGCODE VALIDATION:
   RegCtry values: ['CD'] PASS
   RegCode values: ['BCCO'] PASS
   ListLanguage values: ['French'] PASS

4. KEY